# Data Cleaning

#### Objective
This notebook handles data preparation and cleaning**

We read the three raw ETF csv's from `data/raw/`, clean and validate them then combine them into a single dataset. We also calculate derived columns, and saves the results to `data/processed/`.

In [ ]:
import pandas as pd   
import os              

RAW_DIR       = "../data/raw"  # Define the path to our raw data folder (relative to this notebook)
PROCESSED_DIR = "../data/processed"  # Define the path to our raw data folder (relative to this notebook)
os.makedirs(PROCESSED_DIR, exist_ok=True) # Ensure the raw directory exists

etfs = ["SPY", "QQQ", "VTI"]

print(f"Raw data folder: {os.path.abspath(RAW_DIR)}")
print(f"Processed data folder: {os.path.abspath(PROCESSED_DIR)}")

## 1. Load Raw Data

Load all three raw CSV files from `data/raw/` and store them in a dictionary.  
Print a summary for each file so we know exactly what we are starting with before any cleaning begins.

In [ ]:
raw_data = {} #Dictionary to hold the raw DataFrames

# Keys are ticker names, values are DataFrames

for ticker in etfs:
    path = os.path.join(RAW_DIR, f"{ticker}_raw.csv")
    df = pd.read_csv(path)

    raw_data[ticker] = df

    print(f"{ticker} Raw File Summary")
    print(f"Rows: {len(df)}")
    print(f"Columns: {list(df.columns)}")
    print(f"Nulls per column:")
    print(df.isnull().sum().to_string())
    print(f"\nFirst 5 rows:")
    print(df.head(5).to_string(index=False))
    print()

## 2. Clean Each ETF File

The `clean_etf()` function is applied to all three ETF files.


In [ ]:
def clean_etf(df, ticker):
    """
    Clean a single raw ETF DataFrame.
    
    df: Raw DataFrame loaded from data/raw/
    ticker: Ticker symbol string 

    Returns:
    Cleaned DataFrame with exactly 7 columns matching the data dictionary:
    Date | Open | High | Low | Close | Adj Close | Volume
    """

    df = df.copy()   # Never modify the original DataFrame — always work on a copy
    rows_start = len(df)

    if "Adjusted Close" in df.columns:
        df.rename(columns={"Adjusted Close": "Adj Close"}, inplace=True)

    # Drop the Ticker column 
    if "Ticker" in df.columns:
        df.drop(columns=["Ticker"], inplace=True)

    required_cols = ["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"]
    df = df[[col for col in required_cols if col in df.columns]]
 
    df["Date"] = pd.to_datetime(df["Date"]) #Parse Date as proper datetime type

    # Sort oldest to newest
    df.sort_values("Date", ascending=True, inplace=True)
    df.reset_index(drop=True, inplace=True)

    #Format Date as YYYY-MM-DD string
    df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")

    #Remove duplicate dates 
    dupes = df.duplicated(subset=["Date"]).sum()
    if dupes > 0:
        print(f"  [{ticker}] Removed {dupes} duplicate date(s)")
    df.drop_duplicates(subset=["Date"], keep="first", inplace=True)

    #Drop rows with missing prices
    price_cols  = ["Open", "High", "Low", "Close", "Adj Close"]
    before_drop = len(df)
    df.dropna(subset=price_cols, inplace=True)
    dropped_nulls = before_drop - len(df)
    if dropped_nulls > 0:
        print(f"  [{ticker}] Dropped {dropped_nulls} row(s) with missing prices")

    #Drop rows with zero or negative prices
    before_drop = len(df)
    df = df[(df["Close"] > 0) & (df["Adj Close"] > 0)]
    dropped_zeros = before_drop - len(df)
    if dropped_zeros > 0:
        print(f"  [{ticker}] Dropped {dropped_zeros} row(s) with zero/negative prices")

    df["Volume"] = df["Volume"].fillna(0) #Fill missing Volume with 0

    df.reset_index(drop=True, inplace=True)

    print(f"  [{ticker}] Done: {rows_start} rows → {len(df)} rows "
          f"(removed {rows_start - len(df)} total)")

    return df


cleaned_data = {}
for ticker in etfs:
    print(f"\nCleaning {ticker} ETF.")
    cleaned_data[ticker] = clean_etf(raw_data[ticker], ticker)

print("\nAll ETFs cleaned.")

## 4. Validate Price Logic

- `Low` is the **lowest** price reached that day — `Close` cannot be below it  
- `High` is the **highest** price reached that day — `Close` cannot be above it  

Low <= Close <= High

Any rows that violate this rule are data errors and must be removed before simulations.  
We check all three ETFs and report how many bad rows were found and dropped.

In [ ]:
for ticker in etfs:
    df = cleaned_data[ticker]

    # Flag rows where Close is below Low OR Close is above High
    invalid_mask  = (df["Close"] < df["Low"]) | (df["Close"] > df["High"])
    invalid_count = invalid_mask.sum()

    if invalid_count > 0:
        print(f"[{ticker}] Found {invalid_count} row(s) violating Low <= Close <= High — removing them")
        print(df[invalid_mask][["Date", "Low", "Close", "High"]])

        # Remove the invalid rows and reset the index
        cleaned_data[ticker] = df[~invalid_mask].reset_index(drop=True)
    else:
        print(f"[{ticker}] PASS — all rows satisfy Low <= Close <= High")

## 5. Save Cleaned ETF Files

Save one cleaned CSV per ETF to `data/processed/`.  


In [ ]:
for ticker in etfs:
    df   = cleaned_data[ticker]
    path = os.path.join(PROCESSED_DIR, f"{ticker}_cleaned.csv")

    # index=False prevents pandas from writing a redundant numbered column
    df.to_csv(path, index=False)

    print(f"Saved {ticker}_cleaned.csv — {len(df)} rows | columns: {list(df.columns)}")

## 6. Build a `combined_prices.csv` file 

This file is a merge of all three ETFs using only `Date` and `Adj Close`,  
with `Adj Close` renamed to `SPY_Price`, `QQQ_Price`, and `VTI_Price`.

VTI launched in 2001 so it has no price data before that year.  
An inner join keeps only dates where **all three ETFs have data**,  
so there are no NaN prices in the combined file.  

In [ ]:
# Extract Date and Adj Close from each cleaned file
frames = {}
for ticker in etfs:
    df = cleaned_data[ticker][["Date", "Adj Close"]].copy()

    # Rename Adj Close to the ticker-specific price column name
    df.rename(columns={"Adj Close": f"{ticker}_Price"}, inplace=True)
    frames[ticker] = df

combined = frames["SPY"] \
    .merge(frames["QQQ"], on="Date", how="inner") \
    .merge(frames["VTI"], on="Date", how="inner")  

#Sort by Date ascending
combined.sort_values("Date", ascending=True, inplace=True)
combined.reset_index(drop=True, inplace=True)

print(f"combined_prices.csv (before derived columns)")
print(f"Rows: {len(combined)}")
print(f"Columns: {list(combined.columns)}")
print(f"Date range: {combined['Date'].min()} to {combined['Date'].max()}")
print(f"Nulls: {combined.isnull().sum().to_dict()}")
print(f"\nFirst 3 rows:")
print(combined.head(3).to_string(index=False))

## 7. Final Validation

Check:
1. **Shape**: Expected number of rows and columns
2. **Null counts**: No nulls in price columns 
3. **Price sanity**: All prices are positive numbers
4. **Date format**: All dates match YYYY-MM-DD
5. **No duplicate dates**: Check each date appears exactly once
6. **Sample rows**: Show first and last 3 rows

In [ ]:
print("FINAL VALIDATION — combined_prices.csv")
print(f"\n[1] Shape")
print(f"Rows: {len(combined)}")
print(f"Columns: {len(combined.columns)}  (expected: 4)")
print(f"Column names: {list(combined.columns)}")

print(f"\n[2] Null Counts per Column")
print(combined.isnull().sum().to_string())

# Ensure all prices are positive
print(f"\n[3] Price Sanity — all prices must be greater than 0")
for col in ["SPY_Price", "QQQ_Price", "VTI_Price"]:
    bad    = (combined[col] <= 0).sum()
    status = "PASS" if bad == 0 else f"FAIL: {bad} bad rows"
    print(f"    {col}: {status}")

print(f"\n[4] Date Format Check (expected YYYY-MM-DD)")
bad_dates = combined[~combined["Date"].str.match(r"^\d{4}-\d{2}-\d{2}$")]
status = "PASS" if len(bad_dates) == 0 else f"FAIL — {len(bad_dates)} incorrectly formatted dates"
print(f"{status}")

print(f"\n[5] Duplicate Date Check")
dupes  = combined.duplicated(subset=["Date"]).sum()
status = "PASS" if dupes == 0 else f"FAIL — {dupes} duplicate dates found"
print(f"{status}")

print(f"\n[6] First 3 Rows:")
print(combined.head(3).to_string(index=False))
print(f"\n    Last 3 Rows:")
print(combined.tail(3).to_string(index=False))

## Export to `data/processed/`

In [ ]:
combined_path = os.path.join(PROCESSED_DIR, "combined_prices.csv")
combined.to_csv(combined_path, index=False)

print(f"Saved combined_prices.csv")
print(f"Path: {os.path.abspath(combined_path)}")
print(f"Rows: {len(combined)}")
print(f"Columns: {list(combined.columns)}")